In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [2]:
df = pd.read_csv("cleaned_data1.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      4803 non-null   int64
 1   title   4803 non-null   str  
 2   tags    4803 non-null   str  
dtypes: int64(1), str(2)
memory usage: 5.7 MB


In [4]:
 #Train without a limit to see the true size of your vocabulary
 temp_cv = CountVectorizer(stop_words='english')
 temp_cv.fit(df['tags'])

 print("Total unique words in your dataset:", len(temp_cv.vocabulary_))

Total unique words in your dataset: 76958


In [5]:
# cv = CountVectorizer(max_features = 10000, stop_words = 'english')
# vector = cv.fit_transform(df['tags']).toarray()       # train cv on 'tags' column
# vector

In [6]:
tv = TfidfVectorizer(max_features = 10000, stop_words = 'english')
vector = tv.fit_transform(df['tags']).toarray()       # train cv on 'tags' column
vector

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(4803, 10000))

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vector)
similarity

array([[1.        , 0.0896191 , 0.13544369, ..., 0.03503975, 0.02031178,
        0.02888641],
       [0.0896191 , 1.        , 0.05306061, ..., 0.01930935, 0.01193047,
        0.01502296],
       [0.13544369, 0.05306061, 1.        , ..., 0.0321125 , 0.03047098,
        0.00831801],
       ...,
       [0.03503975, 0.01930935, 0.0321125 , ..., 1.        , 0.03033109,
        0.02805747],
       [0.02031178, 0.01193047, 0.03047098, ..., 0.03033109, 1.        ,
        0.01366794],
       [0.02888641, 0.01502296, 0.00831801, ..., 0.02805747, 0.01366794,
        1.        ]], shape=(4803, 4803))

In [8]:
#Lets write function get name of movie by index
def get_name_by_index(i):
    
    if i < len(df) and i>0:
        return df.loc[i,'title']
    else:
        return 'Index Not found'

In [9]:
a = get_name_by_index(5)
a

'Spider-Man 3'

In [10]:
#Lets write function get index of movie by movie name

# def get_index_from_name(name):
#     found_index = -1
#     for i in df.index:
#         # df.loc[i,'title'] =  Pirates of the Caribbean: At World's End
#         if df.loc[i,'title'].strip().lower() == name.strip().lower():
#             found_index = i
#             break
#     return found_index

In [11]:
# b = get_index_from_name("spider man 3")
# b

In [12]:
def get_index_from_name(name):
    # Normalize user input: lowercase it and strip out all spaces and hyphens
    clean_user_name = name.strip().lower().replace(' ', '').replace('-', '')
    
    # Vectorized pandas match: normalize the dataframe column for comparison
    match = df[df['title'].str.lower().str.replace(' ', '').str.replace('-', '') == clean_user_name]
    
    if not match.empty:
        return match.index[0]
    return -1

In [13]:
b = get_index_from_name("spider man 3")
b

5

In [ ]:
name = input("Enter movie you watched : ")
index = get_index_from_name(name)
# similarity_indexes = list(enumerate(similarity[index]))
# similarity_indexes = sorted(similarity_indexes, key = lambda x: x[1], reverse=True)
# similarity_indexes

if index != -1:
    # Fetch similarity rankings. enumerate() pairs each score with its original movie index like (5, np.float64(0.3635931236853142))

    similarity_indexes = list(enumerate(similarity[index]))

    # The very first item in this sorted list (index 0) will always be the movie itself with a perfect similarity score of 1.0
    similarity_indexes = sorted(similarity_indexes, key=lambda x: x[1], reverse=True)
    
    print(f"\nBecause you watched '{df.loc[index, 'title']}':")
    print("You must watch the following movies:")
    
    # Loop to print top 5 recommendations (skipping index 0 as it's the movie itself)
    for i in range(1, 6):
        movie_idx = similarity_indexes[i][0]
        print(i, ":", get_name_by_index(movie_idx))
else:
    print("Movie Not Found")

In [ ]:
#Lets export similarity 
import pickle as pkl
pkl.dump(similarity, open('similarity.pkl', 'wb'))